# Largest US Companies by Revenue — Web Scraping & Exploratory Analysis

This notebook demonstrates how to use the modular `web_scraping` package to extract, clean, and analyze the **List of largest companies in the United States by revenue** dataset.

> **Keystone Best Practice Note:** Raw HTML dumps (`print(soup)`) are intentionally omitted to keep repository commits lightweight and readable.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Ensure local `src/` package is importable when running from `notebooks/`
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from web_scraping import ScraperConfig, WikipediaCompaniesScraper

## Option A: Live Scraping with `WikipediaCompaniesScraper`
Run the cell below when connected to the internet to fetch the latest table with automatic retry, timeout, footnote cleanup, and deduplication.

In [ ]:
config = ScraperConfig.from_env()
scraper = WikipediaCompaniesScraper(config=config, clean_numbers=True)

# Uncomment to run live network scrape:
# df = scraper.scrape()
# scraper.export_csv(df, "us_largest_companies_by_revenue.csv")

## Option B: Inspect Cleaned Sample Dataset
We can also load the deduplicated and cleaned top-100 dataset stored in `data/sample_us_largest_companies.csv`.

In [ ]:
sample_path = PROJECT_ROOT / "data" / "sample_us_largest_companies.csv"
df = pd.read_csv(sample_path)
df.head(10)

In [ ]:
# Convert Revenue and Employees columns to numeric for summary statistics
df["Revenue_USD_M"] = pd.to_numeric(df["Revenue (USD millions)"].str.replace(",", "", regex=False))
df["Employees_Count"] = pd.to_numeric(df["Employees"].str.replace(",", "", regex=False))

industry_summary = (
    df.groupby("Industry", as_index=False)
    .agg(
        Company_Count=("Name", "count"),
        Total_Revenue_USD_M=("Revenue_USD_M", "sum"),
        Total_Employees=("Employees_Count", "sum"),
    )
    .sort_values("Total_Revenue_USD_M", ascending=False)
    .head(10)
)
industry_summary